In [10]:
import pandas as pd
from sqlalchemy import create_engine
import os

In [11]:
Sever = 'LAPTOP-L3RLU984\SQLEXPRESS'
Database = 'ted_sustentavel'
Driver = 'ODBC Driver 17 for SQL Server'
Database_Con = f'mssql://@{Sever}/{Database}?driver={Driver}'


In [12]:
engine = create_engine(Database_Con)

In [30]:
def carregar_plano_de_contas():
    pasta_plano = r'D:\Arquivos Pessoais\Dados\BusinessCaseT&D\PlanoDeContas'
    
    # Busca qualquer arquivo que contenha "PLANO DE CONTAS" no nome
    arquivos = [f for f in os.listdir(pasta_plano) if "PLANO DE CONTAS" in f.upper()]
    if not arquivos:
        print("ERRO: Arquivo de Plano de Contas não localizado!")
        return
    
    caminho = os.path.join(pasta_plano, arquivos[0])
    print(f"Lendo Plano de Contas em: {caminho}")

    # 1. Leitura bruta sem pular linhas para analisar o conteúdo
    if caminho.endswith('.xlsx'):
        df = pd.read_excel(caminho)
    else:
        df = pd.read_csv(caminho, encoding='latin1', sep=',')

    # 2. Remoção de colunas e linhas que estão COMPLETAMENTE vazias
    # Isso elimina a coluna 'A' vazia e as linhas de respiro no topo
    df = df.dropna(how='all', axis=0).dropna(how='all', axis=1)

    # 3. Localizar onde está o cabeçalho real
    # No seu arquivo, o cabeçalho começa com "COD" ou "CONTAS"
    # Vamos forçar o Pandas a encontrar a linha que contém "COD"
    mask = df.astype(str).apply(lambda x: x.str.contains('COD', case=False)).any(axis=1)
    if mask.any():
        linha_cabecalho = mask.idxmax()
        # Define os dados a partir da linha encontrada e usa a linha como colunas
        df.columns = df.loc[linha_cabecalho]
        df = df.loc[linha_cabecalho + 1:]
    
    # 4. Selecionar e Renomear (Pegamos as 5 primeiras colunas úteis)
    df = df.iloc[:, 0:5]
    df.columns = ['Cod_Nivel4', 'Conta_Nivel4', 'Conta_Nivel3', 'Conta_Nivel2', 'Conta_Nivel1']

    # 5. Limpeza de Segurança
    df = df.dropna(subset=['Cod_Nivel4']) # Remove rodapés ou linhas vazias
    df['Cod_Nivel4'] = df['Cod_Nivel4'].astype(str).str.strip()

    print(f"Sucesso! {len(df)} linhas prontas para o SQL.")
    print(df.head(3)) # Mostra as primeiras 3 linhas no console para conferência

    # 6. Enviar para o SQL
    if not df.empty:
        df.to_sql('Dim_PlanoContas', con=engine, if_exists='replace', index=False)
        print("Tabela Dim_PlanoContas atualizada no SQL Server.")
    else:
        print("Atenção: O processamento resultou em 0 linhas.")

In [21]:
def processar_filiais():
    pasta_dados = r'D:\Arquivos Pessoais\Dados\BusinessCaseT&D\BasedeDados\soFiliais'
    
    for arquivo in os.listdir(pasta_dados):
        if arquivo.endswith('.xlsx') or arquivo.endswith('.csv'):
            caminho_completo = os.path.join(pasta_dados, arquivo)
            print(f"Processando Filial: {arquivo}")
            
            # Leitura flexível
            if arquivo.endswith('.xlsx'):
                df = pd.read_excel(caminho_completo)
            else:
                df = pd.read_csv(caminho_completo, encoding='latin1', sep=',')

            # --- NORMALIZAÇÃO DE COLUNAS ---
            # Forçamos os nomes para minúsculo para facilitar o mapeamento
            df.columns = [c.lower().strip() for c in df.columns]
            
            # Dicionário de mapeamento (ajuste conforme os nomes reais nas suas planilhas)
            mapa_colunas = {
                'código': 'Codigo_Original',
                'codigo': 'Codigo_Original',
                'plano de contas': 'Plano_Contas',
                'data': 'Data_Transacao',
                'data_transacao': 'Data_Transacao',
                'situação': 'Situacao',
                'situacao': 'Situacao',
                'loja': 'Loja_Original'
            }
            df = df.rename(columns=mapa_colunas)

            # --- LIMPEZA DO VALOR ---
            # Remove espaços, pontos de milhar e troca vírgula por ponto
            df['valor'] = df['valor'].astype(str).str.replace(' ', '', regex=False)
            df['valor'] = df['valor'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
            # Remove sinais extras e converte
            df['valor'] = df['valor'].str.replace('+', '', regex=False)
            df['valor'] = pd.to_numeric(df['valor'], errors='coerce')

            # --- EXTRAÇÃO DE METADADOS DO NOME DO ARQUIVO ---
            # Ex: "Filial T&D Niteroi - 2023"
            nome_limpo = arquivo.replace('.xlsx', '').replace('.csv', '')
            if ' - ' in nome_limpo:
                partes = nome_limpo.split(' - ')
                df['Filial_Processada'] = partes[0]
                df['Ano_Referencia'] = partes[1]
            else:
                df['Filial_Processada'] = 'N/A'
                df['Ano_Referencia'] = 0

            # --- TRATAMENTO DE DATA ---
            df['Data_Transacao'] = pd.to_datetime(df['Data_Transacao'], errors='coerce')

            # Selecionar apenas as colunas que existem na tabela Fato_Transacoes
            colunas_finais = ['Codigo_Original', 'Plano_Contas', 'Data_Transacao', 'valor', 
                              'Situacao', 'Loja_Original', 'Filial_Processada', 'Ano_Referencia']
            
            df_final = df[colunas_finais]

            # Enviar para o SQL
            df_final.to_sql('Fato_Transacoes', con=engine, if_exists='append', index=False)

In [31]:
# Execução do Pipeline
if __name__ == "__main__":
    carregar_plano_de_contas
#    processar_filiais()
    print("Processo concluído!")

Processo concluído!
